# Paper 2 — Stage 6c: Filler Activation Extraction

**Runtime: GPU, ~1 hour per model. Run once per model (4 runs). Resumable.**

Extracts ±30-token activation windows at GPT-4o-confirmed FILLER positions, using identical hooks and window logic to Stage 3's revision extraction. These windows are the matched comparison arm that the construct-validity probe (notebook 2) needs — without them there is no matched contrast.

**Prerequisites:** Stage 6a and 6b (`Untitled10.ipynb` cell 0, Phases 1–2) must have produced `rejects/{model}_confirmed_filler.json`. Stage 1 traces must exist.

**Note on completeness:** if any layer's window comes up short near a sequence boundary, that event is dropped entirely rather than zero-padded. So `activations_filler/` has *fewer* events than `rejects/{model}_confirmed_filler.json` lists — do not assume 1:1.


In [ ]:
# ---- Google Drive -----------------------------------------------------------
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
# ---- WHICH MODEL  (run this notebook once per model) ------------------------
# Stage 6c extracts filler activations one model at a time. Set the model here,
# run the whole notebook, then come back and change it. Completed problems are
# skipped on re-run, so interrupting is safe.

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

# The four models, for copy-paste:
#   deepseek-ai/DeepSeek-R1-Distill-Llama-8B
#   deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
#   deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
#   deepseek-ai/DeepSeek-R1-Distill-Qwen-14B

print(f"MODEL_NAME = {MODEL_NAME}")

## Extraction


In [ ]:
# =============================================================================
# TASK 6, PHASE 3 -- FILLER ACTIVATION EXTRACTION (real, original code)
#
# Confirmed real, working code -- not a reconstruction. Extracts activation
# windows at confirmed-filler positions (rejects/{model}_confirmed_filler.json,
# from Phase 1-2), IDENTICAL hooks/window logic to Task 3, writing
# activations_filler/{model}/{problem_id}.npz.
#
# NOTE ON DATA COMPLETENESS: if ANY layer's window comes up short near a
# sequence boundary for a given filler event, that event is dropped entirely
# (not zero-padded) -- see the `ok` flag in the extraction loop. This means
# activations_filler/ has FEWER events than rejects/{model}_confirmed_filler.json
# lists. Do not assume 1:1 correspondence between the two without checking.
#
# NOTE ON METADATA: filler_meta only carries marker, token_idx, matched_text.
# char_start / type_hint / filler_confidence are NOT duplicated into the
# activation metadata -- cross-reference rejects/{model}_confirmed_filler.json
# by matched_text/token position if those fields are needed downstream.
# =============================================================================

import torch, json, gc, numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME  = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"   # <-- change per run
MODEL_SHORT = MODEL_NAME.split("/")[-1]
WINDOW      = 30

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")

In [ ]:
REJECT_BASE = GDRIVE_BASE / "rejects"
TRACE_DIR   = GDRIVE_BASE / "traces" / MODEL_SHORT
OUT_DIR     = GDRIVE_BASE / "activations_filler" / MODEL_SHORT
OUT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)

In [ ]:
model.eval()
N_LAYERS   = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size

store = {}
hooks = []
for i, layer in enumerate(model.model.layers):
    def mk(idx):
        def hook(mod, inp, out):
            h = out[0]
            store[idx] = (h[0] if h.dim() == 3 else h).detach().cpu().float()
        return hook
    hooks.append(layer.register_forward_hook(mk(i)))
print(f"{N_LAYERS} hooks registered")

In [ ]:
def clean_text(t):
    return t.replace("Ġ"," ").replace("Ċ","\n").replace("ĉ","\t")
def get_analysis_text(tr):
    th = clean_text(tr.get("think_content",""))
    cl = clean_text(tr.get("response_clean",""))
    return th if th.strip() else cl
def build_prompt(tr):
    instr = ("Solve the following competition mathematics problem carefully. "
             "Show your reasoning step by step and verify your answer."
             if tr.get("difficulty")=="hard" else
             "Solve the following mathematics competition problem. "
             "Show your working step by step.")
    msgs = [{"role":"user","content":f"{instr}\n\nProblem: {tr['problem']}"}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
def word_to_token(text, widx, plen):
    pref = " ".join(text.split()[:widx])
    return plen + tokenizer(pref, return_tensors="pt",
                            add_special_tokens=False)["input_ids"].shape[1]

with open(REJECT_BASE / f"{MODEL_SHORT}_confirmed_filler.json") as f:
    fillers = json.load(f)

by_problem = {}
for r in fillers:
    by_problem.setdefault(r["problem_id"], []).append(r)

In [ ]:
print(f"{len(fillers)} filler positions across {len(by_problem)} problems")

for i, (pid, entries) in enumerate(sorted(by_problem.items())):
    if (OUT_DIR / f"{pid}.npz").exists():
        continue
    try:
        with open(TRACE_DIR / f"{pid}.json") as f:
            trace = json.load(f)
        analysis = get_analysis_text(trace)
        prompt   = build_prompt(trace)
        plen = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]
        full = tokenizer(prompt + analysis, return_tensors="pt").to(model.device)
        slen = full["input_ids"].shape[1]

        store.clear()
        with torch.no_grad():
            _ = model(**full)

        wins, metas = [], []
        for e in entries:
            t = word_to_token(analysis, e["word_idx"], plen)
            t = max(WINDOW, min(t, slen - WINDOW - 1))
            w = np.zeros((N_LAYERS, 2*WINDOW+1, HIDDEN_DIM), dtype=np.float16)
            ok = True
            for l in range(N_LAYERS):
                sl = store[l][t-WINDOW : t+WINDOW+1]
                if sl.shape[0] != 2*WINDOW+1:
                    ok = False; break
                w[l] = sl.numpy().astype(np.float16)
            if ok:
                wins.append(np.nan_to_num(w, nan=0., posinf=0., neginf=0.))
                metas.append({"marker": e["marker"], "token_idx": int(t),
                              "matched_text": e["matched_text"]})
        store.clear(); torch.cuda.empty_cache(); gc.collect()

        if not wins:
            continue
        np.savez_compressed(OUT_DIR / f"{pid}.npz",
                            filler_windows=np.stack(wins))
        with open(OUT_DIR / f"{pid}_meta.json", "w") as f:
            json.dump({"problem_id": pid, "n_layers": N_LAYERS,
                       "hidden_dim": HIDDEN_DIM, "difficulty": trace.get("difficulty"),
                       "filler_meta": metas}, f, indent=2)
        print(f"[{i+1}/{len(by_problem)}] {pid}: {len(wins)} filler windows")
    except Exception as ex:
        print(f"[{i+1}] {pid} FAILED: {ex}")
        store.clear(); torch.cuda.empty_cache(); gc.collect()

print("Phase 3 done ->", OUT_DIR)